In [ ]:
import pandas as pd

metrics_df = pd.read_csv('Output/metrics_summary_gnn.csv')

summary = (
    metrics_df
    .groupby(["family", "impute_method"])
    [["R2_train", "MAE_train", "R2_test", "MAE_test"]]
    .agg(["mean", "std"])
    .sort_values(("MAE_test", "mean"))
)

print(summary.to_string(float_format="%.2f"))

                      R2_train      MAE_train      R2_test      MAE_test     
                          mean  std      mean  std    mean  std     mean  std
family impute_method                                                         
gnn    mean               0.59 0.07      1.22 0.12    0.37 0.03     1.56 0.07
       most_frequent      0.49 0.03      1.41 0.04    0.34 0.04     1.62 0.07
       median             0.61 0.06      1.21 0.10    0.31 0.04     1.65 0.06
       KNN_imputation     0.65 0.11      1.12 0.20    0.26 0.04     1.68 0.07


In [6]:
import os
import sys

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True, stream=sys.stdout)

import optuna
optuna.logging.set_verbosity(optuna.logging.INFO)

from optuna_objectives import *
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances
import joblib

# ──────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────
N_TRIALS = 50
N_JOBS_OPTUNA = 2
OUT_DIR = 'Output'
TEST_SIZE = 0.2

os.makedirs(OUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────
# Load data
# ──────────────────────────────────────────────
smiles_df = pd.read_csv("Data/cleaned_SMILES_fixed.csv", index_col=0)
dye_device_descs = pd.read_csv("Data/dye_device_pce_without_dupes_new.csv", index_col=0)
pce = dye_device_descs['PCE'].dropna()
dye_device_descs = dye_device_descs.drop('PCE', axis=1).copy().loc[pce.index]
smiles_df = smiles_df.loc[dye_device_descs.index]

print(f'Loaded {len(dye_device_descs)} lines')

fp_cols = [c for c in dye_device_descs.columns if c.startswith("Bit_")]
device_cols = [c for c in dye_device_descs.columns if not c.startswith("Bit_")]

print(f"{len(fp_cols)} fingerprint columns, {len(device_cols)} device features")

groups = smiles_df.loc[dye_device_descs.index, 'SMILES']


# ──────────────────────────────────────────────
# Kennard-Stone split
# ──────────────────────────────────────────────
def kennard_stone_groups(X_df, groups, fp_cols, test_size=0.2):
    unique_dyes = groups.unique()
    n_total = len(groups)
    n_test_target = int(n_total * test_size)

    dye_reps = []
    dye_sizes = {}
    for dye in unique_dyes:
        first_idx = groups[groups == dye].index[0]
        dye_reps.append(X_df.loc[first_idx, fp_cols].values)
        dye_sizes[dye] = (groups == dye).sum()
    dye_reps = np.array(dye_reps)

    dist_matrix = euclidean_distances(dye_reps)
    i, j = np.unravel_index(dist_matrix.argmax(), dist_matrix.shape)
    selected = [i, j]
    n_test_samples = dye_sizes[unique_dyes[i]] + dye_sizes[unique_dyes[j]]

    while n_test_samples < n_test_target:
        remaining = [k for k in range(len(unique_dyes)) if k not in selected]
        if not remaining:
            break
        min_dists = dist_matrix[np.ix_(remaining, selected)].min(axis=1)
        best = remaining[np.argmax(min_dists)]
        selected.append(best)
        n_test_samples += dye_sizes[unique_dyes[best]]

    test_dyes = set(unique_dyes[selected])
    train_dyes = set(unique_dyes) - test_dyes
    test_idx = groups[groups.isin(test_dyes)].index
    train_idx = groups[groups.isin(train_dyes)].index
    return train_idx, test_idx


print("Building Kennard-Stone split...")
train_idx, test_idx = kennard_stone_groups(
    dye_device_descs, groups, fp_cols, test_size=TEST_SIZE
)
n_total = len(dye_device_descs)
train_dyes = set(groups.loc[train_idx])
test_dyes = set(groups.loc[test_idx])
assert len(train_dyes & test_dyes) == 0
print(f"  Train: {len(train_idx)} samples ({len(train_idx)/n_total:.1%}), {len(train_dyes)} dyes")
print(f"  Test:  {len(test_idx)} samples ({len(test_idx)/n_total:.1%}), {len(test_dyes)} dyes")
print("Split is leak-free.\n")

g0 = groups.loc[train_idx]
y0 = pce.loc[train_idx]

# ──────────────────────────────────────────────
# Optimize: HGB + mean for dye-only and device-only
# ──────────────────────────────────────────────
FEATURE_CONFIGS = {
    "hgb_mean_dye_only":    fp_cols,
    "hgb_mean_device_only": device_cols,
}

best_params_all = {}

for key, cols in FEATURE_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Optimizing: {key} ({len(cols)} features)")
    print(f"{'='*60}")

    X0 = dye_device_descs.loc[train_idx, cols]

    best_params = optuna_hgb_study(
        X0, y0, g0,
        n_trials=N_TRIALS,
        n_jobs=N_JOBS_OPTUNA,
        impute_strategy="mean",
    )
    best_params = getattr(best_params, "best_params", best_params)
    best_params_all[key] = best_params

    print(f"Best params for {key}: {best_params}")

# ──────────────────────────────────────────────
# Save
# ──────────────────────────────────────────────
params_path = os.path.join(OUT_DIR, "best_params_ks_dye_device_separate.joblib")
joblib.dump(best_params_all, params_path)
print(f"\nSaved {len(best_params_all)} param sets to {params_path}")

with open(os.path.join(OUT_DIR, "best_params_ks_dye_device_separate.txt"), 'w') as f:
    for key, params in best_params_all.items():
        f.write(f"{key}:\n")
        for k, v in params.items():
            f.write(f"  {k}: {v}\n")
        f.write("\n")

print("Optimization complete.")

c:\Users\anamj\Downloads\PyMol2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Enabling RDKit 2025.03.3 jupyter extensions
Loaded 4351 lines
1024 fingerprint columns, 40 device features
Building Kennard-Stone split...


[I 2026-05-28 07:01:18,938] A new study created in memory with name: Optuna-GradientBoosting-mean


  Train: 3481 samples (80.0%), 1826 dyes
  Test:  870 samples (20.0%), 505 dyes
Split is leak-free.


Optimizing: hgb_mean_dye_only (1024 features)


[I 2026-05-28 07:01:41,650] Trial 0 finished with value: 1.6002135419223225 and parameters: {'max_iter': 500, 'max_depth': 13, 'learning_rate': 0.283641562539681, 'max_leaf_nodes': 86, 'min_samples_leaf': 20, 'l2_regularization': 0.0007666246791818926, 'max_features': 0.6443039359675039}. Best is trial 0 with value: 1.6002135419223225.
[I 2026-05-28 07:01:52,511] Trial 1 finished with value: 1.5881265186452354 and parameters: {'max_iter': 350, 'max_depth': 6, 'learning_rate': 0.09392915220522473, 'max_leaf_nodes': 90, 'min_samples_leaf': 8, 'l2_regularization': 7.800340731201232e-05, 'max_features': 0.9701213476767538}. Best is trial 1 with value: 1.5881265186452354.
[I 2026-05-28 07:02:42,156] Trial 3 finished with value: 1.6173445295486986 and parameters: {'max_iter': 350, 'max_depth': 8, 'learning_rate': 0.04933139998960732, 'max_leaf_nodes': 43, 'min_samples_leaf': 25, 'l2_regularization': 0.09696026040189876, 'max_features': 0.8210834784528391}. Best is trial 1 with value: 1.58812

Best params: {'max_iter': 500, 'max_depth': 15, 'learning_rate': 0.09719534709093201, 'max_leaf_nodes': 99, 'min_samples_leaf': 16, 'l2_regularization': 0.00606083482148799, 'max_features': 0.7272790819087693}
Best params for hgb_mean_dye_only: {'max_iter': 500, 'max_depth': 15, 'learning_rate': 0.09719534709093201, 'max_leaf_nodes': 99, 'min_samples_leaf': 16, 'l2_regularization': 0.00606083482148799, 'max_features': 0.7272790819087693}

Optimizing: hgb_mean_device_only (40 features)


[I 2026-05-28 07:25:48,297] Trial 0 finished with value: 2.0051816197881567 and parameters: {'max_iter': 750, 'max_depth': 4, 'learning_rate': 0.06445671405155219, 'max_leaf_nodes': 76, 'min_samples_leaf': 16, 'l2_regularization': 0.0046347778795818384, 'max_features': 0.7497881175333039}. Best is trial 0 with value: 2.0051816197881567.
[I 2026-05-28 07:25:57,491] Trial 1 finished with value: 1.9502965268958954 and parameters: {'max_iter': 800, 'max_depth': 13, 'learning_rate': 0.06809040731634923, 'max_leaf_nodes': 24, 'min_samples_leaf': 12, 'l2_regularization': 0.00034869659346893535, 'max_features': 0.5252340898173864}. Best is trial 1 with value: 1.9502965268958954.
[I 2026-05-28 07:25:59,401] Trial 2 finished with value: 1.917442673913806 and parameters: {'max_iter': 100, 'max_depth': 16, 'learning_rate': 0.25140533670464693, 'max_leaf_nodes': 77, 'min_samples_leaf': 4, 'l2_regularization': 2.3199765638478597e-06, 'max_features': 0.7450899705551239}. Best is trial 2 with value: 1

Best params: {'max_iter': 500, 'max_depth': 14, 'learning_rate': 0.03273235235658845, 'max_leaf_nodes': 101, 'min_samples_leaf': 1, 'l2_regularization': 1.8553402829739465e-06, 'max_features': 0.5621751019509774}
Best params for hgb_mean_device_only: {'max_iter': 500, 'max_depth': 14, 'learning_rate': 0.03273235235658845, 'max_leaf_nodes': 101, 'min_samples_leaf': 1, 'l2_regularization': 1.8553402829739465e-06, 'max_features': 0.5621751019509774}

Saved 2 param sets to Output\best_params_ks_dye_device_separate.joblib
Optimization complete.


In [7]:
import os
import sys

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True, stream=sys.stdout)

from optuna_objectives import build_final_pipeline_hgb
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd
import numpy as np
import joblib

# ──────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────
N_SPLITS = 100
OUT_DIR = 'Output'
TEST_SIZE = 0.2

os.makedirs(OUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────
# Load data
# ──────────────────────────────────────────────
smiles_df = pd.read_csv("Data/cleaned_SMILES_fixed.csv", index_col=0)
dye_device_descs = pd.read_csv("Data/dye_device_pce_without_dupes_new.csv", index_col=0)
pce = dye_device_descs['PCE'].dropna()
dye_device_descs = dye_device_descs.drop('PCE', axis=1).copy().loc[pce.index]
smiles_df = smiles_df.loc[dye_device_descs.index]

print(f'Loaded {len(dye_device_descs)} lines')

fp_cols = [c for c in dye_device_descs.columns if c.startswith("Bit_")]
device_cols = [c for c in dye_device_descs.columns if not c.startswith("Bit_")]

print(f"{len(fp_cols)} fingerprint columns, {len(device_cols)} device features")

groups = smiles_df.loc[dye_device_descs.index, 'SMILES']


# ──────────────────────────────────────────────
# Random group-based split (20% of samples)
# ──────────────────────────────────────────────
def random_group_split(groups, test_size=0.2, random_state=42):
    rng = np.random.RandomState(random_state)
    unique_dyes = groups.unique()
    n_total = len(groups)
    n_test_target = int(n_total * test_size)

    dye_sizes = {dye: (groups == dye).sum() for dye in unique_dyes}
    shuffled = rng.permutation(unique_dyes)

    test_dyes = set()
    n_test_samples = 0
    for dye in shuffled:
        if n_test_samples >= n_test_target:
            break
        test_dyes.add(dye)
        n_test_samples += dye_sizes[dye]

    train_dyes = set(unique_dyes) - test_dyes
    test_idx = groups[groups.isin(test_dyes)].index
    train_idx = groups[groups.isin(train_dyes)].index
    return train_idx, test_idx


# ──────────────────────────────────────────────
# Build splits
# ──────────────────────────────────────────────
idx_all = dye_device_descs.index
n_total = len(idx_all)
splits = []

print(f"Building {N_SPLITS} random group-based splits...")
for i in range(N_SPLITS):
    tr_idx, te_idx = random_group_split(groups, test_size=TEST_SIZE, random_state=i)
    splits.append((pd.Index(tr_idx), pd.Index(te_idx)))

# Sanity check
for i, (tr, te) in enumerate(splits[:5]):
    train_dyes = set(groups.loc[tr])
    test_dyes  = set(groups.loc[te])
    overlap = train_dyes & test_dyes
    assert len(overlap) == 0, f"Split {i}: {len(overlap)} dyes leaked!"
    print(f'  Split {i}: Train={len(tr)} ({len(tr)/n_total:.1%}), '
          f'Test={len(te)} ({len(te)/n_total:.1%})')
print("All splits are leak-free.\n")


# ──────────────────────────────────────────────
# Load best params
# ──────────────────────────────────────────────
params_path = os.path.join(OUT_DIR, "best_params_ks_dye_device_separate.joblib")
if not os.path.exists(params_path):
    print(f"ERROR: {params_path} not found. Run optimize_dye_device.py first.")
    sys.exit(1)

best_params_all = joblib.load(params_path)
print(f"Loaded {len(best_params_all)} param sets from {params_path}")


# ──────────────────────────────────────────────
# Feature configs (must match optimization keys)
# ──────────────────────────────────────────────
FEATURE_CONFIGS = {
    "hgb_mean_dye_only":    ("dye_only",    fp_cols),
    "hgb_mean_device_only": ("device_only", device_cols),
}


# ──────────────────────────────────────────────
# Train across all splits
# ──────────────────────────────────────────────
metrics_rows = []

for key, (tag, cols) in FEATURE_CONFIGS.items():
    if key not in best_params_all:
        print(f"WARNING: No params found for {key}, skipping.")
        continue

    best_params = best_params_all[key]
    print(f"Training {tag} (HGB + mean) across {N_SPLITS} splits...")
    print(f"  Params: {best_params}")

    for i, (tr_idx, te_idx) in enumerate(splits):
        X_train = dye_device_descs.loc[tr_idx, cols]
        y_train = pce.loc[tr_idx]
        X_test  = dye_device_descs.loc[te_idx, cols]
        y_test  = pce.loc[te_idx]

        pipe = build_final_pipeline_hgb(
            best_params, X_train,
            impute_strategy="mean",
            random_state=i
        )
        pipe.fit(X_train, y_train)

        pred_tr = pipe.predict(X_train)
        pred_te = pipe.predict(X_test)

        metrics_rows.append({
            "model": tag,
            "split": i,
            "R2_train": r2_score(y_train, pred_tr),
            "MAE_train": mean_absolute_error(y_train, pred_tr),
            "R2_test":  r2_score(y_test, pred_te),
            "MAE_test": mean_absolute_error(y_test, pred_te),
        })

        if i % 10 == 0:
            print(f"  Split {i}: R2_test={r2_score(y_test, pred_te):.4f}, "
                  f"MAE_test={mean_absolute_error(y_test, pred_te):.4f}")


# ──────────────────────────────────────────────
# Save results
# ──────────────────────────────────────────────
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(os.path.join(OUT_DIR, "metrics_dye_device_separate.csv"), index=False)
print(f"\nSaved metrics ({len(metrics_df)} rows)")


# ──────────────────────────────────────────────
# Print summary
# ──────────────────────────────────────────────
summary = (
    metrics_df
    .groupby("model")
    [["R2_train", "MAE_train", "R2_test", "MAE_test"]]
    .agg(["mean", "std"])
    .sort_values(("MAE_test", "mean"))
)
print("\n" + summary.to_string(float_format="%.2f"))
print("\nTraining complete.")

Loaded 4351 lines
1024 fingerprint columns, 40 device features
Building 100 random group-based splits...
  Split 0: Train=3481 (80.0%), Test=870 (20.0%)
  Split 1: Train=3479 (80.0%), Test=872 (20.0%)
  Split 2: Train=3481 (80.0%), Test=870 (20.0%)
  Split 3: Train=3481 (80.0%), Test=870 (20.0%)
  Split 4: Train=3479 (80.0%), Test=872 (20.0%)
All splits are leak-free.

Loaded 2 param sets from Output\best_params_ks_dye_device_separate.joblib
Training dye_only (HGB + mean) across 100 splits...
  Params: {'max_iter': 500, 'max_depth': 15, 'learning_rate': 0.09719534709093201, 'max_leaf_nodes': 99, 'min_samples_leaf': 16, 'l2_regularization': 0.00606083482148799, 'max_features': 0.7272790819087693}
  Split 0: R2_test=0.6614, MAE_test=1.0573
  Split 10: R2_test=0.6338, MAE_test=1.1612
  Split 20: R2_test=0.6292, MAE_test=1.1291
  Split 30: R2_test=0.6090, MAE_test=1.2026
  Split 40: R2_test=0.6114, MAE_test=1.1752
  Split 50: R2_test=0.6431, MAE_test=1.1793
  Split 60: R2_test=0.6457, MAE_

In [8]:
"""
baseline_residuals.py — Residual plots for the baseline trio.
Run after baseline.py or standalone (re-trains the models).
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

from optuna_objectives import build_preprocessor

# ──────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────
OUT_PATH = "Output/baseline_residuals.png"
TEST_SIZE = 0.2

plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# ──────────────────────────────────────────────
# Load data
# ──────────────────────────────────────────────
print("Loading data...")
smiles_df = pd.read_csv("Data/cleaned_SMILES_fixed.csv", index_col=0)
dye_device_descs = pd.read_csv("Data/dye_device_pce_without_dupes_new.csv", index_col=0)
pce = dye_device_descs['PCE'].dropna()
dye_device_descs = dye_device_descs.drop('PCE', axis=1).copy().loc[pce.index]
smiles_df = smiles_df.loc[dye_device_descs.index]

fp_cols = [c for c in dye_device_descs.columns if c.startswith("Bit_")]
device_cols = [c for c in dye_device_descs.columns if not c.startswith("Bit_")]
groups = smiles_df.loc[dye_device_descs.index, 'SMILES']


# ──────────────────────────────────────────────
# Kennard-Stone split
# ──────────────────────────────────────────────
def kennard_stone_groups(X_df, groups, fp_cols, test_size=0.2):
    unique_dyes = groups.unique()
    n_total = len(groups)
    n_test_target = int(n_total * test_size)

    dye_reps = []
    dye_sizes = {}
    for dye in unique_dyes:
        first_idx = groups[groups == dye].index[0]
        dye_reps.append(X_df.loc[first_idx, fp_cols].values)
        dye_sizes[dye] = (groups == dye).sum()
    dye_reps = np.array(dye_reps)

    dist_matrix = euclidean_distances(dye_reps)
    i, j = np.unravel_index(dist_matrix.argmax(), dist_matrix.shape)
    selected = [i, j]
    n_test_samples = dye_sizes[unique_dyes[i]] + dye_sizes[unique_dyes[j]]

    while n_test_samples < n_test_target:
        remaining = [k for k in range(len(unique_dyes)) if k not in selected]
        if not remaining:
            break
        min_dists = dist_matrix[np.ix_(remaining, selected)].min(axis=1)
        best = remaining[np.argmax(min_dists)]
        selected.append(best)
        n_test_samples += dye_sizes[unique_dyes[best]]

    test_dyes = set(unique_dyes[selected])
    train_dyes = set(unique_dyes) - test_dyes
    test_idx = groups[groups.isin(test_dyes)].index
    train_idx = groups[groups.isin(train_dyes)].index
    return train_idx, test_idx


print("Building Kennard-Stone split...")
train_idx, test_idx = kennard_stone_groups(
    dye_device_descs, groups, fp_cols, test_size=TEST_SIZE
)

# ──────────────────────────────────────────────
# Train 3 models
# ──────────────────────────────────────────────
feature_sets = {
    "Dyes: Morgan Fingerprints": fp_cols,
    "Devices: Extracted Device Features": device_cols,
    "Dye-Device Pairs: Morgan Fingerprints + Device Features": fp_cols + device_cols,
}

TRAIN_COLOR = "#3A7EBF"
TEST_COLOR = "#E8A838"

results = {}
for label, cols in feature_sets.items():
    print(f"Training: {label}...")
    X_train = dye_device_descs.loc[train_idx, cols]
    X_test = dye_device_descs.loc[test_idx, cols]
    Y_train = pce.loc[train_idx]
    Y_test = pce.loc[test_idx]

    preprocessor = build_preprocessor(X_train, impute_strategy=None)
    hgb = HistGradientBoostingRegressor(random_state=42)
    pipe = make_pipeline(preprocessor, hgb)
    pipe.fit(X_train, Y_train)

    pred_train = pipe.predict(X_train)
    pred_test = pipe.predict(X_test)

    results[label] = {
        "Y_train": Y_train, "Y_test": Y_test,
        "pred_train": pred_train, "pred_test": pred_test,
        "resid_train": Y_train.values - pred_train,
        "resid_test": Y_test.values - pred_test,
    }
    # ── Shared axis limits across all panels ──
    all_pred = np.concatenate([np.concatenate([r["pred_train"], r["pred_test"]])
                            for r in results.values()])
    all_resid = np.concatenate([np.concatenate([r["resid_train"], r["resid_test"]])
                                for r in results.values()])

    x_min, x_max = np.min(all_pred) - 0.5, np.max(all_pred) + 0.5
    y_min, y_max = np.min(all_resid) - 0.5, np.max(all_resid) + 0.5
    # Symmetrize y-axis around zero
    y_lim = max(abs(y_min), abs(y_max))


# ──────────────────────────────────────────────
# Create residual plots
# ──────────────────────────────────────────────
print("Generating residual plots...")

fig = plt.figure(figsize=(10, 10))

ax1 = fig.add_axes([0.08, 0.55, 0.40, 0.38])
ax2 = fig.add_axes([0.55, 0.55, 0.40, 0.38])
ax3 = fig.add_axes([0.30, 0.08, 0.40, 0.38])

axes_map = {
    "Dyes: Morgan Fingerprints": ax1,
    "Devices: Extracted Device Features": ax2,
    "Dye-Device Pairs: Morgan Fingerprints + Device Features": ax3,
}

for label, ax in axes_map.items():
    r = results[label]

    # Scatter residuals vs predicted
    ax.scatter(r["pred_train"], r["resid_train"],
               c=TRAIN_COLOR, alpha=0.3, s=10, edgecolors='none',
               label='Train', zorder=2)
    ax.scatter(r["pred_test"], r["resid_test"],
               c=TEST_COLOR, alpha=0.5, s=10, edgecolors='none',
               label='Test', zorder=3)

    # Zero line
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.8, alpha=0.7, zorder=1)

    # Labels
    ax.set_xlabel("Predicted PCE", fontsize=10)
    ax.set_ylabel("Residual (Measured − Predicted)", fontsize=10)
    ax.tick_params(labelsize=8)
    ax.set_title(label, fontsize=10, fontweight='bold', pad=10)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(-y_lim, y_lim)

    # Stats box
    mae_te = np.mean(np.abs(r["resid_test"]))
    mean_resid = np.mean(r["resid_test"])
    std_resid = np.std(r["resid_test"])
    stats_text = (
        f"Test residuals:\n"
        f"  Mean = {mean_resid:.2f}\n"
        f"  Std = {std_resid:.2f}\n"
        f"  MAE = {mae_te:.2f}"
    )
    ax.text(0.03, 0.97, stats_text, transform=ax.transAxes,
            fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#DEB887',
                      edgecolor='#C4A36E', alpha=0.9))

    ax.legend(fontsize=8, loc='lower right', framealpha=0.9)

fig.savefig(OUT_PATH, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved to {OUT_PATH}")

Loading data...
Building Kennard-Stone split...
Training: Dyes: Morgan Fingerprints...
Training: Devices: Extracted Device Features...
Training: Dye-Device Pairs: Morgan Fingerprints + Device Features...
Generating residual plots...
Saved to Output/baseline_residuals.png


In [1]:
import joblib
import pandas as pd

shap_data = joblib.load("Output/SHAP/shap_values.joblib")
pred_test = shap_data['pred_test']
X_test_index = shap_data['X_test_index']

pred_series = pd.Series(pred_test, index=X_test_index)
bins = pd.cut(pred_series, 3)
print(bins.cat.categories)

c:\Users\anamj\Downloads\PyMol2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IntervalIndex([(-0.652, 3.366], (3.366, 7.372], (7.372, 11.378]], dtype='interval[float64, right]')
